# NumCompute-Stream Demo

This demo loads a CSV dataset using the custom I/O pipeline, splits it into chunks, trains a streaming model incrementally, logs metrics, and visualises performance over time.

In [ ]:
import os, sys
import matplotlib
matplotlib.use("Agg")
sys.path.insert(0, os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'demo' else sys.path.insert(0, os.getcwd())
import numpy as np
from numcompute_stream.io import read_csv_numpy, chunk_array
from numcompute_stream import Pipeline, StandardScaler, Imputer, DecisionTreeClassifier, EnsembleClassifier, StreamTrainer
from numcompute_stream.visualise import plot_metric_over_time, compare_models, plot_predictions_vs_ground_truth


## Load CSV data
The included student-performance CSV is used as a numeric demonstration dataset. The final column is treated as a classification target after thresholding.

In [ ]:
from pathlib import Path

# Works whether this notebook is run from the project root or from demo/.
candidate_paths = [
    Path('dataset/test_student_performance_data.csv'),
    Path('demo/dataset/test_student_performance_data.csv'),
]
csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find test_student_performance_data.csv in demo/dataset/')

data = read_csv_numpy(str(csv_path), skip_header=True)
data = data[~np.isnan(data).all(axis=1)]
X = data[:, :-1].astype(float)
raw_y = data[:, -1].astype(float)
y = (raw_y >= np.nanmedian(raw_y)).astype(int)
X.shape, y.shape

## Train incrementally on chunks

In [ ]:
tree_pipe = Pipeline([
    ('impute', Imputer(strategy='mean')),
    ('scale', StandardScaler()),
    ('model', DecisionTreeClassifier(max_depth=5, min_samples_split=4))
])
forest_pipe = Pipeline([
    ('impute', Imputer(strategy='mean')),
    ('scale', StandardScaler()),
    ('model', EnsembleClassifier(n_estimators=5, max_depth=5, random_state=42))
])

tree_trainer = StreamTrainer(tree_pipe)
forest_trainer = StreamTrainer(forest_pipe)

chunk_size = 128
latest_y = latest_tree_pred = latest_forest_pred = None
for X_chunk, y_chunk in chunk_array(X, y, chunk_size):
    tree_trainer.fit_chunk(X_chunk, y_chunk)
    forest_trainer.fit_chunk(X_chunk, y_chunk)
    latest_y = y_chunk
    latest_tree_pred = tree_pipe.predict(X_chunk)
    latest_forest_pred = forest_pipe.predict(X_chunk)

tree_trainer.logs


## Visualise streaming metrics

In [ ]:
plot_metric_over_time(tree_trainer.logs['cumulative_accuracy'],
                      title='Decision tree cumulative accuracy',
                      ylabel='Accuracy', show=False)


In [ ]:
compare_models(tree_trainer.logs['cumulative_accuracy'],
               forest_trainer.logs['cumulative_accuracy'],
               labels=('Decision tree', 'Ensemble'), show=False)


In [ ]:
plot_predictions_vs_ground_truth(latest_y, latest_forest_pred, show=False)


## Final comparison

In [ ]:
print('Tree final cumulative accuracy:', tree_trainer.logs['cumulative_accuracy'][-1])
print('Ensemble final cumulative accuracy:', forest_trainer.logs['cumulative_accuracy'][-1])
print('Tree memory log:', tree_trainer.logs['memory_bytes'][-3:])
print('Ensemble memory log:', forest_trainer.logs['memory_bytes'][-3:])
